In [1]:
%run /home/mls07/speculative-decoding/notebooks/model.ipynb

import time

from metrics import _sync, autoregresivno, perplexity

EVAL_FILE = "/data/evaluation/prefix-evaluation.txt"
N_SEQUENCES = 20
MAX_LEN = 256
PROMPT_LEN = 32
GEN_TOKENS = 50
REPEATS = 3

True
Wed Aug 12 20:58:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.10              Driver Version: 570.86.10      CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:90:00.0 Off |                    0 |
| N/A   32C    P0             51W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

あなたの名前は何ですか。____
A. なまえは何ですか。
B. なまえは何です。
C. なまえはなんですか。
D. なまえはなんです。
答案: なまえはなんですか。

下列属于非银行金融机构的是____。 Ⅰ．货币经纪公司 Ⅱ．金融租赁公司 Ⅲ．信托公司 Ⅳ．金融资产管理公司
A. Ⅰ、Ⅱ、Ⅲ、�
cuda:0


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

あなたの名前は何ですか？

A. あなたの名前は何ですか？
B. あなたの名前は何ですか？
C. あなたの名前は何ですか？
D. あなたの名前は何ですか？

あなたが何を言っていないか理解してください。

A. あなたが何を言っていないか理解してください。
B. あなたが何を言っていないか理解してください。
C. あなたが何を言っていないか理解してください。
D. あなたが何を言っていないか理解してください。

あなた
あなたの名前は何ですか。____
A. なまえ
B. なに
C. なん
D. なにか
答案:

なに

男性,67岁。因进行性呼吸困难10天人院。既往有肺气肿病史。动脉血气分析:PH 7.30, PaO240mmHg,PaCO265mmHg,最适宜的吸氧浓度是
A. 51%-60%
B. 31%-40%
C. 41%-50%
D. 21%-30%
答案:

D

你は名前を書く＿＿どうしましたか。____
A. 场合
B. 場面
C. 様子
D. 般
答案:

場合

下列关于证券公司从事自营业务时需要控制的风险中，不属于合规风险的是


In [2]:
def load_eval_sequences(path, n, max_len):
    with open(path, "r", encoding="utf-8") as f:
        paragraphs = [p for p in f.read().split("\n\n") if p.strip()]

    sequences = []
    for p in paragraphs[:n]:
        ids = tokenizer(p, return_tensors="pt", add_special_tokens=False).input_ids[:, :max_len]
        if ids.shape[1] >= 2:
            sequences.append(ids)
    return sequences


In [3]:
@torch.no_grad()
def top1_accuracy(model, sequences):
    device = next(model.parameters()).device
    correct, total = 0, 0
    for ids in sequences:
        ids = ids.to(device)
        preds = model(ids).logits[0, :-1].argmax(dim=-1)
        targets = ids[0, 1:]
        correct += (preds == targets).sum().item()
        total += targets.numel()
    return correct / total

In [4]:
@torch.no_grad()
def measure_inference_time(model, prompt_ids, gen_tokens, repeats):
    device = next(model.parameters()).device
    ids = prompt_ids.to(device)

    autoregresivno(model, ids, gen_tokens)  # warmup
    _sync(device)

    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        autoregresivno(model, ids, gen_tokens)
        _sync(device)
        times.append(time.perf_counter() - t0)

    avg = sum(times) / len(times)
    return avg, gen_tokens / avg

In [5]:
def benchmark(name, model, sequences, prompt_ids, gen_tokens, repeats):
    ppl = perplexity(model, sequences)
    acc = top1_accuracy(model, sequences)
    seconds, tokens_per_s = measure_inference_time(model, prompt_ids, gen_tokens, repeats)
    return {
        "model": name,
        "perplexity": ppl,
        "top1_accuracy": acc,
        "s_per_generation": seconds,
        "tokens_per_s": tokens_per_s,
    }

In [6]:
def print_results(results):
    header = f"{'model':<10}{'perplexity':>12}{'top1_acc':>10}{'tokens/s':>10}"
    print(header)
    print("-" * len(header))
    for r in results:
        print(f"{r['model']:<10}{r['perplexity']:>12.3f}{r['top1_accuracy']:>10.3f}{r['tokens_per_s']:>10.2f}")


In [7]:
sequences = load_eval_sequences(EVAL_FILE, N_SEQUENCES, MAX_LEN)
prompt_ids = sequences[0][:, :PROMPT_LEN]

results = []
for name, model in [("teacher", teacher), ("student", student)]:
    model.eval()
    results.append(benchmark(name, model, sequences, prompt_ids, GEN_TOKENS, REPEATS))

print_results(results)

model       perplexity  top1_acc  tokens/s
------------------------------------------
teacher          6.399     0.584     26.17
student         13.277     0.490     51.09
